[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/thomas_precession/thomas_precession.ipynb)

# Thomas Precession and the Muon's Anomalous Magnetic Moment

This example computes how a spinning particle's axis turns when its velocity changes direction, though nothing twists it. Two boosts in different directions do not make a single boost: they leave a turn behind. Carried around any closed loop of velocities, a particle's axes come back turned by the area the loop encloses in the space of velocities, a space curved like a saddle; on a circular orbit this is Thomas precession. The same turning, with a magnetic moment added, is what the muon g−2 experiment measures: in its storage ring the muon's spin runs ahead of its momentum, at a rate set by how far its magnetic moment departs from two units.

A boost is modelled as a rotor, and the velocities a particle can have as the lab's direction in time under every boost, a hyperboloid. The straight boost between two velocities is a square root, `(head * tail).square_root()`, and a spin carried along a path of velocities is turned by the product of those rotors; when the path closes, the product returns the velocity to itself and only turns the axes across it. In a field, one bivector turns the particle's whole frame, its velocity and its spin together.

In the notation of hyperbolic geometry the velocities form the hyperboloid model of the hyperbolic plane, $\{u : u \cdot u = 1,\ u^0 > 0\}$, with rapidity as distance, and the turn around a closed loop reads as the holonomy of its Levi-Civita connection, the curvature integrated over the enclosed region, $\int K\,dA = -A$ for $K = -1$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import STA
from examples.animation import save_figure, show_animation
from examples.relativity.thomas_precession import render

In [ ]:
# Spacetime, with space positive, `x+y+z+t-`. Velocities, events and directions are antivectors, intrinsic
# like all geometry; fields, and the planes spinning gyroscopes turn in, are bivectors.
ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
Rotor = ga.gatype.rotor()
# The observer at rest: one unit along the lab's time, the direction across its slice of space mv.t.
time = -mv.t.dual()                                                         # [] Antivector
# The unit volume.
pseudoscalar = mv.xyzt                                                      # [] Pseudoscalar
duration_ms = 50

## 1. Two boosts make a turn

A rocket at rest fires its engine along its own x axis until it moves at some rapidity, then along its own y axis by the same amount. A single boost straight from rest would reach the same final velocity, but the rocket's axes do not arrive as that boost would carry them: they are turned. Composing the two boosts and undoing the straight one, `rest_boost(rocket >> time).reverse() * rocket`, leaves a rotor that keeps the direction in time: a pure turn, the Wigner rotation.

The animation draws velocities in the Poincaré disk: rest is the centre, the rim is the speed of light, and the straight boosts are the lines of the hyperbolic plane, arcs meeting the rim square. At each corner of the triangle of velocities the rocket's x axis appears as the observer at rest reads it; the faint arrow at the last corner is where the straight boost from rest would have left it. The turn between them is the triangle's area, its angles' shortfall from half a turn, against the way round the triangle.

In matrix notation the composition reads as the product of two Lorentz matrices, which factors into a symmetric boost matrix and an orthogonal rotation, $\Lambda_y \Lambda_x = B\,R(\theta)$, with $\tan(\theta/2) = \tanh^2(\eta/2)$ for two equal perpendicular boosts of rapidity $\eta$.

In [ ]:
# The rocket's two equal boosts, their rapidity swelling and shrinking between two values, and the
# samples along each side of the triangle of velocities.
n_boost_frames = 96
min_rapidity, max_rapidity = 0.8, 2.0
n_edge_samples = 32

In [ ]:
def boost(direction: Antivector, rapidity: np.ndarray) -> Rotor:
    """The pure boost by `rapidity` along a unit spatial direction: a turn in the plane of the direction
    and time, their join, which the dual turns into a generator."""
    return ((direction & time).dual() * (rapidity / 2)).exp()                   # [...] Rotor


def rest_boost(velocity: Antivector) -> Rotor:
    """The pure boost carrying the lab's direction in time to `velocity`."""
    return (velocity * time.inverse()).square_root()                            # [...] Rotor


def transport(tail: Antivector, head: Antivector) -> Rotor:
    """The pure boost from velocity `tail` to velocity `head`, in their plane."""
    return (head * tail.inverse()).square_root()                                # [...] Rotor


def geodesic(tail: Antivector, head: Antivector, fractions: np.ndarray) -> Antivector:
    """The velocities along the pure boost from `tail` to `head`, at each fraction of the way."""
    return (transport(tail, head).log() * fractions).exp() >> tail             # [fractions, ...] Antivector


def two_boosts(rapidities: np.ndarray, fractions: np.ndarray) -> Iterator[tuple[Antivector, Antivector, Antivector]]:
    """For each rapidity: the triangle of velocities a rocket passes through, boosting along its own x
    and then its own y, and straight back to rest; its corners; and the rocket's x axis at each corner,
    as the observer at rest reads it."""
    for rapidity in rapidities:
        first = boost(mv.x.dual(), rapidity)                                    # [] Rotor
        rocket = first * boost(mv.y.dual(), rapidity)                           # [] Rotor
        corner, velocity = first >> time, rocket >> time                        # [] Antivector each
        loop = concatenate([geodesic(time, corner, fractions), geodesic(corner, velocity, fractions),
                            geodesic(velocity, time, fractions)])               # [3 * samples] Antivector
        # The rocket's x axis at each corner, after the pure boost back to rest.
        axes = stack([mv.x.dual(), rest_boost(corner).reverse() >> (first >> mv.x.dual()),
                      rest_boost(velocity).reverse() >> (rocket >> mv.x.dual())])   # [3] Antivector
        yield loop, stack([time, corner, velocity]), axes


swell = (1 - np.cos(np.linspace(0, 2 * np.pi, n_boost_frames, endpoint=False))) / 2
rapidities = min_rapidity + (max_rapidity - min_rapidity) * swell              # [frames]
fractions = np.linspace(0, 1, n_edge_samples)                                    # [samples]
# The turn the two boosts leave: their product after the straight boost is undone.
rocket = boost(mv.x.dual(), rapidities) * boost(mv.y.dual(), rapidities)         # [frames] Rotor
wigner = rest_boost(rocket >> time).reverse() * rocket                           # [frames] Rotor
show_animation(render.animate_two_boosts(two_boosts(rapidities, fractions)), "thomas_two_boosts", duration_ms)

## 2. Thomas precession on a circular orbit

A particle going round a circle at constant speed changes its velocity all the time, and its velocity goes round a circle in the space of velocities. Nothing twists it, so its spin, the plane a gyroscope on it turns in, is carried from each velocity to the next by the straight boost between them, `transport(tail, head)`: the rocket's two kicks, made many and small. Each kick leaves a small turn, and the turns add up. After one orbit the spin, as the observer at rest reads it, is turned against the orbit by the area of the cap of velocities inside the circle, $2\pi(\gamma - 1)$ at Lorentz factor $\gamma$: Thomas precession. The three orbits, at Lorentz factors 1.25, 1.5 and 1.75, turn their spins back by a quarter, a half and three quarters of a turn; the faint arrows keep the starting direction.

In the three-vector notation of special relativity the precession reads as the rate $\boldsymbol\omega_T = \frac{\gamma^2}{\gamma + 1}\,\frac{\mathbf a \times \mathbf v}{c^2}$, which integrates to $-2\pi(\gamma - 1)$ per orbit.

In [ ]:
# Lorentz factors of the orbits, the steps around each, and the steps between frames.
gammas = np.array([1.25, 1.5, 1.75])
n_orbit_steps = 180
orbit_stride = 2

In [ ]:
def lab_view(velocity: Antivector, spin: Bivector) -> Bivector:
    """A spin of the frame moving at `velocity`, as the observer at rest reads it: after the pure
    boost back to rest."""
    return (time * velocity.inverse()).square_root() >> spin                    # [...] Bivector


def about(axis: Antivector) -> Bivector:
    """The plane of space that turns about a spatial direction, as a rate: the unit volume times the
    plane the direction sweeps through time."""
    return pseudoscalar * (axis & time).dual()                                  # [...] Bivector


def carried(velocities: Antivector, spin: Bivector) -> Iterator[Bivector]:
    """A spin carried along a path of velocities `[steps + 1, ...]` by the pure boost between each
    velocity and the next, torque-free: the spin at each velocity."""
    yield spin
    for tail, head in zip(velocities[:-1], velocities[1:]):
        spin = transport(tail, head) >> spin                                    # [...] Bivector
        yield spin


def orbits(velocities: Antivector, spin: Bivector, stride: int) -> Iterator[tuple[Antivector, Bivector]]:
    """Every `stride`-th velocity along the orbits, with the carried spin as the observer at rest reads it."""
    for velocity, transported in islice(zip(velocities, carried(velocities, spin)), None, None, stride):
        yield velocity, lab_view(velocity, transported)


phases = np.linspace(0, 2 * np.pi, n_orbit_steps + 1)
# Moving along y from x, counterclockwise.
directions = (mv.xy * (-phases[:, None] / 2)).exp() >> mv.y.dual()               # [steps + 1, 1] Antivector
orbit_velocities = boost(directions, np.arccosh(gammas)) >> time                 # [steps + 1, orbits] Antivector
# A spin along x, turning about it, in each orbit's first frame.
start_spin = rest_boost(orbit_velocities[0]) >> about(mv.x.dual())               # [orbits] Bivector
show_animation(render.animate_orbits(orbit_velocities, orbits(orbit_velocities, start_spin, orbit_stride)),
               "thomas_orbits", duration_ms)

## 3. A gyroscope bouncing in a box

Thomas precession needs no circle. A gyroscope bounces round a square box, from the middle of each wall to the next. In flight its velocity holds and nothing turns. At a wall the momentum along the wall holds and the momentum across it reverses, so in the space of velocities each bounce runs from one corner of a square of momenta to the next, lifted onto the hyperboloid by `on_hyperboloid`. The spin turns only in the bounces, a little each time and always the same way: after each round trip, by the area the square of momenta encloses on the hyperboloid. Any loop of velocities turns the spin by its area; the circle was only the simplest.

In [ ]:
# The box's half width, the gyroscope's momentum per unit mass along each axis, the frames of each
# flight and the steps of each bounce, and the round trips shown.
half_width = 1.0
momentum = 0.8
n_flight_frames = 10
n_bounce_steps = 10
n_loops = 3

In [ ]:
def on_hyperboloid(spatial: Antivector) -> Antivector:
    """The velocity whose spatial part is `spatial`: the unit direction in time above it."""
    return spatial + time * (1 + spatial.squared()).square_root()               # [...] Antivector


def bouncing(positions: Antivector, velocities: Antivector, spin: Bivector) -> Iterator[tuple[Antivector, Bivector]]:
    """Each position, with the carried spin as the observer at rest reads it."""
    for position, velocity, transported in zip(positions, velocities, carried(velocities, spin)):
        yield position, lab_view(velocity, transported)


# Quarter turns from one before the first wall to one past the last.
quarters = (mv.xy * (-np.arange(-1, 5) * np.pi / 4)).exp()                      # [6] Rotor
walls = quarters >> (half_width * mv.x.dual())                                  # [6] Antivector
corners = quarters >> (momentum * (mv.x.dual() + mv.y.dual()))                  # [6] Antivector
# Each flight runs from the wall before to its wall; each bounce from its momentum to the next.
starts, ends = walls[:-2], walls[1:-1]                                          # [4] Antivector each
flights = starts + (ends - starts) * np.linspace(0, 1, n_flight_frames)[:, None]   # [flight, 4] Antivector
before, after = corners[1:-1], corners[2:]                                      # [4] Antivector each
bounces = on_hyperboloid(before + (after - before) * np.linspace(0, 1, n_bounce_steps)[:, None])   # [bounce, 4] Antivector
flight_velocities = on_hyperboloid(before).broadcast_to((n_flight_frames, 4))   # [flight, 4] Antivector
# The round trip, wall by wall, repeated.
positions = concatenate([part[:, wall] for wall in range(4)
                         for part in (flights, ends.broadcast_to((n_bounce_steps, 4)))] * n_loops)   # [frames] Antivector
box_velocities = concatenate([part[:, wall] for wall in range(4)
                              for part in (flight_velocities, bounces)] * n_loops)                 # [frames] Antivector
box_spin = rest_boost(box_velocities[0]) >> about(mv.x.dual())                  # [] Bivector
show_animation(render.animate_box(half_width, ends, bouncing(positions, box_velocities, box_spin)),
               "thomas_box", duration_ms)

## 4. The muon's anomalous magnetic moment

A charged particle with a magnetic moment turns twice over in a magnetic field: its momentum goes round the orbit, and its spin precesses about the field. For a magnetic moment of exactly two units, g = 2, the two turn together. The muon's moment is larger by its anomaly, (g − 2)/2 ≈ 0.00116592, and its spin runs ahead of its momentum by a full turn every 1/(anomaly × γ) orbits. The Muon g−2 experiment at Fermilab stores muons in a ring and watches that lead: a muon's decay positron flies preferentially along its spin, so the count in the detectors rises and falls at the anomalous rate.

One bivector carries both turns. The field is a bivector, its magnetic part a plane of space and its electric part a plane through time, and `bmt_generator` turns the muon's whole frame: `frame >> time` is its velocity and `frame >> about(mv.y.dual())` its spin. The force is the field read along the velocity, `velocity & field`, and on the velocity the plane of the particle's own slice of space, `-velocity.dual()`, with that force acts as the field itself does, so the anomaly cancels, so the orbit is the Lorentz force's whatever the moment; on the spin it subtracts the Thomas precession of section 2 from the magnetic precession. The anomaly here is exaggerated to 0.1, so that the spin, orange, laps the momentum, grey, every three orbits.

In the four-tensor index notation of relativistic electrodynamics the turning reads as the Lorentz force $\frac{du^\mu}{d\tau} = \frac{q}{m} F^\mu{}_\nu u^\nu$ beside the Bargmann–Michel–Telegdi equation $\frac{ds^\mu}{d\tau} = \frac{q}{m}\left(\frac{g}{2} F^{\mu}{}_{\nu} s^\nu + \left(\frac{g}{2} - 1\right) u^\mu F_{\nu\lambda} u^\nu s^\lambda\right)$, two equations for what the bivector does in one.

In [ ]:
# The anomaly, exaggerated so that the spin laps the momentum in a few orbits, at the Lorentz factor
# of section 5; the magnetic field, and a ring without electric field; the orbits shown, the steps of
# each, the steps between frames, and the ring's samples.
anomaly = 0.1
ring_gamma = np.sqrt(1 + 1 / anomaly)
magnetic = 1.0
ring_electric = 0.0
n_turns = 3
n_turn_steps = 120
ring_stride = 3
n_ring_samples = 128

In [ ]:
def bmt_generator(field: Bivector, velocity: Antivector, anomaly: float) -> Bivector:
    """The bivector turning the frame of a particle of magnetic anomaly `anomaly`, `g / 2 - 1`: its
    velocity by the Lorentz force and its spin by the Bargmann–Michel–Telegdi equation."""
    force = velocity & field                                                    # [...] Vector
    # The particle's own slice of space, which reads its velocity as one: minus the velocity's dual, as
    # for the lab's `time`.
    rest_slice = -velocity.dual()                                               # [...] Vector
    # On the velocity the plane of the slice and the force acts as the field does, so the anomaly
    # cancels there.
    return (1 + anomaly) * field - anomaly * (rest_slice ^ force)              # [...] Bivector


def ring_field(event: Antivector, magnetic: float, electric: np.ndarray) -> Bivector:
    """A storage ring's field at an event: a uniform magnetic field in the orbit's plane, and an
    electric field of strength `electric` pointing away from the ring's axis. The magnetic field points
    along minus z, so that it bends a positive charge counterclockwise."""
    position = event - (mv.t & event) * time                                    # [...] Antivector
    outward = position / position.squared().square_root()                      # [...] Antivector
    # A magnetic field along a direction is the plane turning about it; an electric field, the dual of
    # the plane the direction sweeps through time.
    return -magnetic * about(mv.z.dual()) + electric * (outward & time).dual()     # [...] Bivector


def bmt_step(event: Antivector, frame: Rotor, magnetic: float, electric: np.ndarray, anomaly: float,
             proper_step: np.ndarray) -> tuple[Antivector, Rotor]:
    """One midpoint step in proper time of a particle's event and frame in the ring's field."""
    rate = bmt_generator(ring_field(event, magnetic, electric), frame >> time, anomaly)   # [...] Bivector
    halfway_frame = (rate * (proper_step / 4)).exp() * frame                   # [...] Rotor
    halfway_event = event + (frame >> time) * (proper_step / 2)                 # [...] Antivector
    halfway_rate = bmt_generator(ring_field(halfway_event, magnetic, electric), halfway_frame >> time, anomaly)
    return event + (halfway_frame >> time) * proper_step, (halfway_rate * (proper_step / 2)).exp() * frame


def evolve(event: Antivector, frame: Rotor, magnetic: float, electric: np.ndarray, anomaly: float,
           proper_step: np.ndarray) -> Iterator[tuple[Antivector, Rotor]]:
    """The particle's event and frame at every step, without end."""
    while True:
        yield event, frame
        event, frame = bmt_step(event, frame, magnetic, electric, anomaly, proper_step)


def ring_radius(gamma: np.ndarray, magnetic: float, electric: np.ndarray) -> np.ndarray:
    """The radius of the circular orbit at Lorentz factor `gamma`, where the magnetic pull inwards less
    the electric push outwards turns the momentum."""
    speed = np.sqrt(1 - 1 / gamma**2)
    return gamma * speed**2 / (speed * magnetic - electric)


def ring_start(gamma: np.ndarray, magnetic: float, electric: np.ndarray) -> tuple[Antivector, Rotor]:
    """A particle on the circular orbit at Lorentz factor `gamma`, at x and moving along y: its event,
    and its frame, the pure boost from rest, so that its spin, turning about y, lies along the
    momentum."""
    event = mv.x.dual() * ring_radius(gamma, magnetic, electric)           # [...] Antivector
    frame = boost(mv.y.dual(), np.arccosh(gamma))                          # [...] Rotor
    return event, frame


def ring(gamma: float, magnetic: float, electric: float, anomaly: float, n_turns: int, n_turn_steps: int,
         stride: int) -> Iterator[tuple[Antivector, Antivector, Bivector]]:
    """Every `stride`-th step of a particle's turns around the ring: its event, velocity, and spin as
    the observer at rest reads it."""
    radius = ring_radius(gamma, magnetic, electric)
    proper_period = 2 * np.pi * radius / np.sqrt(gamma**2 - 1)
    event, frame = ring_start(gamma, magnetic, electric)
    states = evolve(event, frame, magnetic, electric, anomaly, proper_period / n_turn_steps)
    for event, frame in islice(states, 0, n_turns * n_turn_steps + 1, stride):
        velocity = frame >> time                                                # [] Antivector
        yield event, velocity, lab_view(velocity, frame >> about(mv.y.dual()))


radius = ring_radius(ring_gamma, magnetic, ring_electric)
outline = (mv.xy * (-np.linspace(0, 2 * np.pi, n_ring_samples) / 2)).exp() >> (radius * mv.x.dual())   # [samples] Antivector
show_animation(render.animate_ring(outline, ring(ring_gamma, magnetic, ring_electric, anomaly, n_turns, n_turn_steps, ring_stride)),
               "thomas_storage_ring", duration_ms)

## 5. The magic momentum

A real storage ring also needs electric fields, to keep the muons focused, and an electric field seen by a moving muon is partly magnetic: it turns the spin too, and would spoil the measurement. Its share of the anomalous rate is proportional to $a - 1/(\gamma^2 - 1)$, which vanishes at one Lorentz factor, $\gamma = \sqrt{1 + 1/a}$: the magic Lorentz factor, 29.3 for the muon, at a momentum of 3.094 GeV/c, where the rings at Brookhaven and Fermilab run. The muons below are started along their momentum on circular orbits through a radial electric field of each strength, at a Lorentz factor of 2, grey, and at the magic Lorentz factor, orange; after a fixed time in the lab, the angle the spin has gained gives the anomalous rate, over the anomaly times the magnetic field. At the magic Lorentz factor it stays at one, whatever the electric field.

In the three-vector notation of the g−2 experiments the anomalous rate reads as $\boldsymbol\omega_a = \frac{e}{m}\left[a\mathbf B - \left(a - \frac{1}{\gamma^2 - 1}\right)\frac{\boldsymbol\beta \times \mathbf E}{c}\right]$.

In [ ]:
# The radial electric fields swept, at a Lorentz factor of 2 and the magic one, over a fixed time in the lab.
electric_fields = np.linspace(-0.3, 0.3, 7)
sweep_gammas = np.array([2.0, np.sqrt(1 + 1 / anomaly)])
lab_duration = 10.0
n_sweep_steps = 200

In [ ]:
def anomalous_angle(velocity: Antivector, spin: Bivector) -> Scalar:
    """The angle in the orbit's plane from the plane a spin along the momentum turns in to the spin, as
    the observer at rest reads both."""
    motion = velocity - (mv.t & velocity) * time                                # [...] Antivector
    direction = motion / motion.squared().square_root()                         # [...] Antivector
    # A spin along the momentum turns about its direction.
    along = about(direction)                                                    # [...] Bivector
    # The product of two such planes is the rotor through their angle.
    return (lab_view(velocity, spin) * along.inverse()).log().scalar_product(mv.xy)   # [...] Scalar


event, frame = ring_start(sweep_gammas[:, None], magnetic, electric_fields)       # [gammas, fields] Antivector, Rotor
proper_step = lab_duration / (sweep_gammas[:, None] * n_sweep_steps)              # [gammas, 1]
*_, (event, frame) = islice(evolve(event, frame, magnetic, electric_fields, anomaly, proper_step), n_sweep_steps + 1)
# The spins started along the momentum: the angle each has gained, as a rate over the anomaly times the field.
ratios = anomalous_angle(frame >> time, frame >> about(mv.y.dual())) / (anomaly * magnetic * lab_duration)   # [gammas, fields] Scalar
save_figure(render.draw_magic(electric_fields / magnetic, ratios), "thomas_magic");

In [ ]:
# checks
# Two boosts leave a pure turn, by the closed form for equal perpendicular boosts.
np.testing.assert_allclose(((wigner >> time) - time).kernel, 0.0, atol=1e-12)
angle = 2 * wigner.log().scalar_product(mv.xy).kernel[..., 0]
np.testing.assert_allclose(angle, -2 * np.arctan(np.tanh(rapidities / 2) ** 2), atol=1e-12)
# Around a circular orbit the spin turns back by the cap's area, 2 pi (gamma - 1).
*_, orbit_end = carried(orbit_velocities, start_spin)
expected = (mv.xy * (np.pi * (gammas - 1))).exp() >> about(mv.x.dual())
np.testing.assert_allclose((lab_view(orbit_velocities[-1], orbit_end) - expected).kernel, 0.0, atol=5e-3)
# In the box, each round trip turns it back by the area the square of momenta encloses on the hyperboloid.
square_area = 4 * (2 * momentum * np.arcsinh(momentum / np.sqrt(1 + momentum**2))
                   - np.arctan(momentum**2 / np.sqrt(1 + 2 * momentum**2)))
*_, (_, box_end) = bouncing(positions, box_velocities, box_spin)
expected = (mv.xy * (n_loops * square_area / 2)).exp() >> about(mv.x.dual())
np.testing.assert_allclose((box_end - expected).kernel, 0.0, atol=5e-2)
# At the magic Lorentz factor the electric field drops out of the anomalous rate.
np.testing.assert_allclose(ratios.kernel[1, :, 0], 1.0, atol=1e-3)